<a href="https://colab.research.google.com/github/jashwanth-cse/Jashwanth-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_03_ETL_Pandas_APIs/Day_03_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install requests --quiet
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
raw_df=pd.read_csv('/content/drive/MyDrive/messy_sales_data.csv')
print(f"Dataset loaded:{raw_df.shape[0]}students,{raw_df.shape[1]} columns")
print(raw_df.columns.tolist())

Dataset loaded:30students,9 columns
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


In [7]:
print("Data Quality Diagnosis Report")
print("-------------------------------")
print("No of missing values in each column:")
print(raw_df.isnull().sum())
print("Duplicate Rows:",raw_df.duplicated().sum())
print("-------------------------------")
print("Data Types:")
print(raw_df.dtypes)

print("Unique Categories:",raw_df['category'].unique())
print("Sample customer name",raw_df['customer_name'].dropna().unique()[:8])
print("Sample order date:",raw_df['order_date'].unique()[:6])

Data Quality Diagnosis Report
-------------------------------
No of missing values in each column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Duplicate Rows: 0
-------------------------------
Data Types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
Unique Categories: ['Electronics' 'Accessories' nan]
Sample customer name ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
Sample order date: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [8]:
copy_df=raw_df.copy()
print("Working copy created")

print("raw_df is untouched- we can always reset by running df=raw_df.copy()")

Working copy created
raw_df is untouched- we can always reset by running df=raw_df.copy()


Median -Data's center value(middle value)


In [9]:
# Fix #1: Handle Missing Values

print("Before fixing nulls:",raw_df.isnull().sum().sum(),'total missing values')


#Fix missing quantity -fill with null customer names

raw_df['customer_name'].fillna('Unknown customer',inplace=True)

median_qty=raw_df['quantity'].median()
raw_df['quantity'].fillna(median_qty,inplace=True)
print("Filled missing quantity with median",median_qty)

print("After fixing nulls:",raw_df.isnull().sum().sum(),'total missing values')

Before fixing nulls: 7 total missing values
Filled missing quantity with median 2.0
After fixing nulls: 2 total missing values


In [10]:
print("Before duplication",len(raw_df),"rows")
#Show duplicate rows
print("Duplicate rows ",raw_df.duplicated().sum())
print(raw_df[raw_df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])

#Remove exact duplicate rows
raw_df.drop_duplicates(inplace=True)
print("After deduplication",len(raw_df),"rows")
print("Rows removed",len(copy_df)-len(raw_df))

Before duplication 30 rows
Duplicate rows  0
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After deduplication 30 rows
Rows removed 0


In [11]:
print("Sample dates before parsing")
print(raw_df['order_date'].head(8).tolist())
raw_df['order_date']=pd.to_datetime(
    raw_df['order_date'],
    dayfirst=False,
    errors='coerce' #if parsing fails put NaT
)

raw_df['year']=raw_df['order_date'].dt.year
raw_df['month']=raw_df['order_date'].dt.month
raw_df['month_name']=raw_df['order_date'].dt.strftime("%B")

print("\nSample dates after parsing")
print(raw_df[['order_date','year','month','month_name']].head(10))

Sample dates before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

Sample dates after parsing
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
5        NaT     NaN    NaN        NaN
6 2024-01-12  2024.0    1.0    January
7 2024-01-13  2024.0    1.0    January
8 2024-01-15  2024.0    1.0    January
9 2024-01-15  2024.0    1.0    January


In [12]:
#Fix 4 Standardization
print("Before standardization",raw_df['customer_name'].unique()[:6])

raw_df['customer_name']=(
    raw_df['customer_name']
    .str.strip()
    .str.title()
)

print("Before keyboard rows with Electronics category ")
wrong_mask=(raw_df['product']=='keyboard')&(raw_df['category']=="Electronics")

print(raw_df[wrong_mask],[['product','category']])

raw_df.loc[wrong_mask,'category']='Accessories'

Before standardization ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
Before keyboard rows with Electronics category 
Empty DataFrame
Columns: [order_id, customer_name, product, category, quantity, unit_price, order_date, city, sales_rep, year, month, month_name]
Index: [] [['product', 'category']]


In [13]:
raw_df['quantity']=pd.to_numeric(raw_df['quantity'],errors='coerce')
raw_df['unit_price']=pd.to_numeric(raw_df['unit_price'],errors='coerce')
raw_df['revenue']=raw_df['quantity']*raw_df['unit_price']
print('Revenue column created:')
print(raw_df[['customer_name','product','quantity','unit_price','revenue']].head())
print(f"\n Total Revenue across all orders: rs{raw_df['revenue'].sum():,.0f}")

Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

 Total Revenue across all orders: rs818,000


In [20]:
print("POST CLEANING VALIDATION REPORT")
print("Original rows:", len(copy_df))
print("Cleaned rows:", len(raw_df))
print("Rows removed:", len(copy_df) - len(raw_df))
print("Missing values:", raw_df.isnull().sum().sum())
print("Duplicates:", raw_df.duplicated().sum())
print("Date nulls:", raw_df['order_date'].isnull().sum())
print("Revenue NaN:", raw_df['revenue'].isnull().sum())
print("Categories:", sorted(raw_df['category'].dropna().unique()))

all_clean = (
    raw_df.isnull().sum().sum() == 0 and
    raw_df.duplicated().sum() == 0
)

print("Dataset fully clean:", all_clean)

POST CLEANING VALIDATION REPORT
Original rows: 30
Cleaned rows: 30
Rows removed: 0
Missing values: 10
Duplicates: 0
Date nulls: 2
Revenue NaN: 0
Categories: ['Accessories', 'Electronics']
Dataset fully clean: False


In [16]:
product_rev=(
    raw_df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)

print("Revenue by Product")

Revenue by Product


In [21]:
API_KEY='a601765a8cc2881c45eeabdc5a9696a0'
BASE_URL='https://api.openweathermap.org/data/2.5/weather'
CITIES=["Mumbai","Hyderabad","Chennai","Rajapalayam","Delhi","Sivakasi"]


In [22]:
# ============================================================
# CELL 14 — EXTRACT: Call API for each city
# ============================================================
import requests;
def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,      # City name query parameter
        'appid': API_KEY,   # Authentication key
        'units': 'metric'   # Returns temperature in Celsius
    }
    # params is a dictionary — requests will encode it into the URL:
    # ?q=Mumbai&appid=KEY&units=metric

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        # requests.get() sends an HTTP GET request to BASE_URL
        # timeout=10 — wait max 10 seconds; raise error if no response

        if response.status_code == 200:
            # status_code 200 = HTTP OK = request was successful
            data = response.json()
            # .json() parses the JSON text body into a Python dictionary

            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
                # .get('visibility', 0) — safe access: returns 0 if key missing
                # // 1000 — convert meters to kilometers (integer division)
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None
    # try/except handles errors gracefully — one bad city doesn't crash the loop


# Call API for all cities
print('Calling Weather API...')
weather_records = []
# Empty list — we will append one dict per city

for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        # .append() adds the dict to our list
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

Calling Weather API...
  Fetching: Mumbai... 31.0°C, Haze
  Fetching: Hyderabad... 35.2°C, Haze
  Fetching: Chennai... 32°C, Few Clouds
  Fetching: Rajapalayam...  ERROR 404 for Rajapalayam: city not found
 FAILED
  Fetching: Delhi... 31.1°C, Overcast Clouds
  Fetching: Sivakasi... 31.7°C, Overcast Clouds

Successfully fetched: 5/6 cities


1. What are the three stages of ETL? Describe each stage using an example from today's sales dataset.
2. A DataFrame has 500 rows. After calling df.dropna(), it has 412 rows. What does this tell you?
3. Write code to remove duplicates from df where 'same row' means same customer_name AND same product.
4. What is the difference between fillna(9) and fillna(df['col'].median()) ? When would you prefer each?
5. Write Python code to call the weather API for 'Delhi' and print the temperature in Celsius.
Q6: What does response.status_code == 200 mean? What should you do when the code is 401?

1. ** Extraction:** extracting data & storing it in sales db
**Transform:** clean the data by filling null values & handling fields without customer names.And fixing date formats.
**Load:**
After cleaning the data the cleaned data is stored to the database for analytics/processing

2. 88 records with null values is removed.

3. df = df.drop_duplicates(
    subset=['customer_name', 'product']
)

4. fillna(9) replaces missing values with a fixed value (9).
Use it when a default or constant value makes sense in the dataset.

fillna(df['col'].median()) replaces missing values with the column’s middle value.
Use it for numerical data to keep the data more realistic and balanced.

5. 200 status code: It means OK , the response is got from the server success.

401 is unauthorized & its due to wrong api key or the request may contain expired  api key.



In [24]:
print("----------------MINI PROJECT---------------------")
print("----------------Weather Data ETL Project---------")
# requests is used to send API requests to the weather server
import requests

# pandas is used for data cleaning and working with tables
import pandas as pd


# Your OpenWeatherMap API key
# Replace this with your own API key



# City name for which weather data is needed
city = "Chennai"


# API URL with city name and API key
# units=metric gives temperature in Celsius
url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"


# Send GET request to the API
response = requests.get(url)


# Convert JSON response into Python dictionary
data = response.json()


# Print full raw API response (optional for understanding)
print("Raw API Data:")
print(data)


# Create a dictionary with only required fields
weather_data = {

    # Store city name
    "City": [data["name"]],

    # Store weather condition
    "Weather": [data["weather"][0]["main"]],

    # Store weather description
    "Description": [data["weather"][0]["description"]],

    # Store temperature
    "Temperature": [data["main"]["temp"]],

    # Store humidity percentage
    "Humidity": [data["main"]["humidity"]],

    # Store wind speed
    "Wind Speed": [data["wind"]["speed"]]
}


# Convert dictionary into Pandas DataFrame
df = pd.DataFrame(weather_data)


# Print original DataFrame
print("\nOriginal DataFrame:")
print(df)


# Remove duplicate rows if any
df = df.drop_duplicates()


# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())


# Fill missing temperature values with mean temperature
# (Useful when working with multiple records)
df["Temperature"] = df["Temperature"].fillna(df["Temperature"].mean())


# Convert weather text to uppercase for consistency
df["Weather"] = df["Weather"].str.upper()


# Print cleaned DataFrame
print("\nCleaned DataFrame:")
print(df)


# Save cleaned data into CSV file
df.to_csv("cleaned_weather_data.csv", index=False)


# Print success message
print("\nCleaned weather data saved successfully!")

----------------MINI PROJECT---------------------
----------------Weather Data ETL Project---------
Raw API Data:
{'coord': {'lon': 80.2785, 'lat': 13.0878}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02n'}], 'base': 'stations', 'main': {'temp': 32, 'feels_like': 39, 'temp_min': 30.69, 'temp_max': 32.2, 'pressure': 1009, 'humidity': 78, 'sea_level': 1009, 'grnd_level': 1009}, 'visibility': 6000, 'wind': {'speed': 7.72, 'deg': 190}, 'clouds': {'all': 20}, 'dt': 1779981914, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1779927101, 'sunset': 1779973253}, 'timezone': 19800, 'id': 1264527, 'name': 'Chennai', 'cod': 200}

Original DataFrame:
      City Weather Description  Temperature  Humidity  Wind Speed
0  Chennai  Clouds  few clouds           32        78        7.72

Missing Values:
City           0
Weather        0
Description    0
Temperature    0
Humidity       0
Wind Speed     0
dtype: int64

Cleaned DataFrame:
      City Weather 